<a href="https://colab.research.google.com/github/wmasfoe/md-editor-models/blob/master/notebooks/train_and_release_t4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="在 Colab 中打开"/></a>

> ⚠️ 本 notebook 使用 **master** 分支的 Qwen3 + Adapter 发布链路。

# 🚀 md-editor 端侧专属小模型: Google Colab L4/T4 矩阵微调与发布

本 Notebook 可在 **Google Colab (推荐 L4 / T4 GPU)** 上一键完成：
1. **环境自检**：检测 NVIDIA L4 / T4 GPU 与硬件加速状态
2. **多模型矩阵微调**：支持单独训练 **0.5B Lite** / **1.5B Standard**，或 **一键全矩阵全自动打包**
3. **自动量化与增量聚合**：转换为 `Q4_K_M` GGUF，并自动增量合并到同一个 `manifest.json`
4. **自动发布**：一键推送到 GitHub Releases 同一版本 Tag 下

In [1]:
#@title ⚙️ [1/4] 配置训练参数与发布模式
#@markdown 推荐模式说明：
#@markdown - 「Lite 完整矩阵（推荐: 约25分钟）」：快速发布 Lite 0.6B 基座及三个轻量 Adapter，适合日常验证与发版；
#@markdown - 「Standard 完整矩阵 (Qwen3-1.7B)」：发布 Standard 1.7B 基座及三个 Adapter；
#@markdown - 「Lite + Standard 双矩阵串行 (长耗时)」：顺序发布全部 8 个资产；
#@markdown - 单任务模式：用于补跑特定失败资产。

mode = "Lite 完整矩阵（推荐: 约25分钟)" #@param ["Lite 完整矩阵（推荐: 约25分钟)", "Standard 完整矩阵 (Gemma 4 E2B)", "Standard 完整矩阵 (Qwen3-1.7B)", "Lite + Standard 双矩阵串行 (长耗时)", "base", "gec adapter", "completion adapter", "distill adapter", "legacy 完整模型(旧版兼容)"]
version_tag = "v1.3.1" #@param {type:"string"}
# 下列两个字段仅在非全矩阵模式下使用：
tier = "lite" #@param ["lite", "standard"]
base_model = "Qwen/Qwen3-0.6B" #@param {type:"string"}
branch = "master" #@param {type:"string"}

# 验证 GPU 状态 (需显示 L4 / T4 / A100)
!nvidia-smi

print(f"🎯 模式: {mode}")
print(f"🏷️ 版本: {version_tag}")
if "Lite 完整矩阵" in mode:
    print("📦 将发布 Lite (Qwen3-0.6B) 的 Base 与 3 个任务 Adapter (约25分钟)")
elif "Standard 完整矩阵 (Gemma 4 E2B)" in mode:
    print("📦 将发布 Standard (Gemma 4 E2B) 的 Base 与 3 个任务 Adapter (LIMA极简对齐)")
elif "Standard 完整矩阵" in mode:
    print("📦 将发布 Standard (Qwen3-1.7B) 的 Base 与 3 个任务 Adapter")
elif "双矩阵" in mode:
    print("📦 将依次发布 Lite 与 Standard 完整资产矩阵 (长耗时)")
else:
    print(f"📦 Tier: {tier} | Base: {base_model}")


🎯 已选择模式: 1.5B (Standard - L4约15分钟)
🏷️ 版本标签:   v1.0.0
Tue Sep  1 14:28:21 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   51C    P8             12W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N

In [ ]:
#@title 📦 [2/4] 克隆/更新仓库并预装全套依赖环境
import os
if not os.path.exists('/content/md-editor-models'):
    !git clone -b "{branch}" https://github.com/wmasfoe/md-editor-models.git /content/md-editor-models
else:
    !git -C /content/md-editor-models fetch origin "{branch}" && git -C /content/md-editor-models checkout "{branch}" && git -C /content/md-editor-models reset --hard "origin/{branch}"

%cd /content/md-editor-models
# 卸载 Colab 自带的旧版不兼容 torchao (0.10.0)
!pip uninstall -y torchao
!pip install -q trl peft pangu datasets transformers accelerate sentencepiece gguf protobuf huggingface_hub
print("✅ 仓库与依赖就绪")
!git -C /content/md-editor-models rev-parse --short HEAD
!git -C /content/md-editor-models status --short --branch

Cloning into '/content/md-editor-models'...
remote: Enumerating objects: 119, done.
remote: Counting objects: 100% (119/119), done.
remote: Compressing objects: 100% (86/86), done.
remote: Total 119 (delta 50), reused 95 (delta 28), pack-reused 0 (from 0)
Receiving objects: 100% (119/119), 799.91 KiB | 22.85 MiB/s, done.
Resolving deltas: 100% (50/50), done.
/content/md-editor-models
From https://github.com/wmasfoe/md-editor-models
 * branch            master     -> FETCH_HEAD
Already up to date.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 992.6/992.6 kB 58.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.1/559.1 kB 52.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 118.5/118.5 kB 14.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 118.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 47.1 MB/s eta 0:00:00


In [ ]:
#@title 🧪 [可选] 在 Google Colab 免费 GPU 上即时评测 Gemma 4 裸机中文纠错表现
#@markdown 无需任何训练，直接加载 Gemma 4 裸机，测试其在零微调下对错别字（如“因该”、“布署”）的纠错表现及无错句子的克制力：

eval_model_id = "google/gemma-4-E2B-it" #@param ["google/gemma-4-E2B-it", "google/gemma-4-e2b-it", "google/gemma-4-E4B-it", "Qwen/Qwen3-1.7B"]
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

device = "cuda:0" if torch.cuda.is_available() else "cpu"
dtype = torch.bfloat16 if (torch.cuda.is_available() and torch.cuda.is_bf16_supported()) else torch.float16

print(f"🚀 正在加载 {eval_model_id} 进行零微调实时评测 (目标设备: {device})...")
tokenizer = AutoTokenizer.from_pretrained(eval_model_id, trust_remote_code=True)

# 显式使用单卡映射并捕获 fallback，避免 accelerate 跨设备切片时将非持久化缓冲区遗留在 meta 设备
try:
    model = AutoModelForCausalLM.from_pretrained(
        eval_model_id,
        torch_dtype=dtype,
        device_map={"": device},
        trust_remote_code=True
    )
except Exception:
    model = AutoModelForCausalLM.from_pretrained(
        eval_model_id,
        torch_dtype=dtype,
        low_cpu_mem_usage=False,
        trust_remote_code=True
    ).to(device)

# 🛡️ 鲁棒性防线：自动检测并物化任何遗留在 meta 上的非持久化缓冲区与参数（如 Gemma 4 音频/相对位置编码）
for name, buf in model.named_buffers():
    if getattr(buf, "is_meta", False):
        parent_name, buf_name = name.rsplit(".", 1) if "." in name else ("", name)
        parent = model.get_submodule(parent_name) if parent_name else model
        parent.register_buffer(buf_name, torch.zeros(buf.shape, dtype=buf.dtype, device=device), persistent=False)

for name, param in model.named_parameters():
    if getattr(param, "is_meta", False):
        parent_name, param_name = name.rsplit(".", 1) if "." in name else ("", name)
        parent = model.get_submodule(parent_name) if parent_name else model
        parent.register_parameter(param_name, torch.nn.Parameter(torch.zeros(param.shape, dtype=param.dtype, device=device)))

model.eval()

test_cases = [
    "今天我们因该去公园散步。",
    "后端服务的布署配置已经完成。",
    "我们在项目中使用了 LoRA 微调和 Q4_K_M 量化的 GGUF 模型。",
    "接口调用的 paramater 参数需要重新核对。",
    "这是一篇完全写对的学术论文引言，没有任何语病和错字。"
]

sys_prompt = '你是一个 Markdown 语法与错别字纠错器。找出文本中的错别字与修改建议，仅输出紧凑 JSON 格式 [[start, end, "原词", "建议词"]]。如果文本完全无错，请严格仅输出 []，禁止输出任何多余解释。'

print()
print("================ 零微调测试结果 ================")
for idx, text in enumerate(test_cases):
    messages = [
        {"role": "system", "content": sys_prompt},
        {"role": "user", "content": f"请检查：{text}"}
    ]
    inputs = tokenizer.apply_chat_template(messages, tokenize=True, add_generation_prompt=True, return_tensors="pt", return_dict=True)
    inputs = {k: v.to(device) for k, v in inputs.items()}
    with torch.no_grad():
        outputs = model.generate(**inputs, max_new_tokens=64, do_sample=False)
    input_len = inputs["input_ids"].shape[1]
    resp = tokenizer.decode(outputs[0][input_len:], skip_special_tokens=True).strip()
    print(f"[{idx+1}] 输入: {text}")
    print(f"    输出: {resp}")
    print()


In [ ]:
#@title 🎯 [路线 A 验证] 500 条样本极速微调 Gemma 4 LoRA 并在 Colab 即时测试特殊 Token (<|task_gec_zh|>)
#@markdown 保持客户端协议 100% 不变：构建 500 条轻量样本微调 Gemma 4 LoRA 适配器，并在裸机与客户端真实请求（无 System Prompt，直接 `<|task_gec_zh|>文本`）下测试！

import os
import subprocess
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

target_base = "google/gemma-4-E2B-it"
lora_output_dir = "output/gemma4-e2b-gec-lora"
device = "cuda" if torch.cuda.is_available() else "cpu"
dtype = torch.bfloat16 if (torch.cuda.is_available() and torch.cuda.is_bf16_supported()) else torch.float16

print("=================================================================")
print("🚀 [1/4] 构建 500 条高质量格式对齐数据集 (含 40% 真实无错负样本)...")
print("=================================================================")
subprocess.run(["python3", "scripts/build_dataset.py", "--mode", "tiny-format", "--max_samples", "500"], check=True)

print("\n=================================================================")
print(f"🚀 [2/4] 在当前 GPU 上微调轻量 LoRA (r=8, assistant_only_loss=True)...")
print("=================================================================")
train_cmd = [
    "python3", "train_sft.py",
    "--model_name_or_path", target_base,
    "--train_file", "data/train.jsonl",
    "--val_file", "data/val.jsonl",
    "--output_dir", lora_output_dir,
    "--task", "gec",
    "--num_train_epochs", "1",
    "--batch_size", "16",
    "--learning_rate", "1e-4",
    "--lora_r", "8",
    "--lora_alpha", "16",
    "--weight_decay", "0.01",
    "--lora_dropout", "0.1",
    "--assistant_only_loss"
]
subprocess.run(train_cmd, check=True)

print("\n=================================================================")
print(f"🚀 [3/4] 加载微调好的 LoRA Adapter 并进行实测验证...")
print("=================================================================")
tokenizer = AutoTokenizer.from_pretrained(lora_output_dir, trust_remote_code=True)
base_model = AutoModelForCausalLM.from_pretrained(
    target_base,
    torch_dtype=dtype,
    device_map={"": device},
    trust_remote_code=True
)

# 自动物化 meta 缓冲区
for name, buf in base_model.named_buffers():
    if getattr(buf, "is_meta", False):
        parent_name, buf_name = name.rsplit(".", 1) if "." in name else ("", name)
        parent = base_model.get_submodule(parent_name) if parent_name else base_model
        parent.register_buffer(buf_name, torch.zeros(buf.shape, dtype=buf.dtype, device=device), persistent=False)

for name, param in base_model.named_parameters():
    if getattr(param, "is_meta", False):
        parent_name, param_name = name.rsplit(".", 1) if "." in name else ("", name)
        parent = base_model.get_submodule(parent_name) if parent_name else base_model
        parent.register_parameter(param_name, torch.nn.Parameter(torch.zeros(param.shape, dtype=param.dtype, device=device)))

model = PeftModel.from_pretrained(base_model, lora_output_dir).to(device)
model.eval()

test_cases = [
    ("今天我们因该去公园散步。", "拼音近音选词翻车 (因该 -> 应该)"),
    ("后端服务的布署配置已经完成。", "技术术语混淆错字 (布署 -> 部署)"),
    ("我们在项目中使用了 LoRA 微调和 Q4_K_M 量化的 GGUF 模型。", "技术术语完全正确句子 (测试防过度编辑/防误报)"),
    ("接口调用的 paramater 参数需要重新核对。", "英文参数拼写错误 (paramater -> parameter)"),
    ("这是一篇完全写对的学术论文引言，没有任何语病和错字。", "纯中文完全正确散文 (测试负样本抑制)")
]

print("\n" + "=" * 65)
print("🎯 路线 A 实测结果 (客户端真实协议: 仅发送 <|task_gec_zh|>，无系统提示词)")
print("=" * 65)
for idx, (text, desc) in enumerate(test_cases):
    prompt = f"<|task_gec_zh|>{text}"
    messages = [{"role": "user", "content": prompt}]
    inputs = tokenizer.apply_chat_template(messages, tokenize=True, add_generation_prompt=True, return_tensors="pt", return_dict=True)
    inputs = {k: v.to(device) for k, v in inputs.items()}
    with torch.no_grad():
        outputs = model.generate(**inputs, max_new_tokens=64, do_sample=False)
    input_len = inputs["input_ids"].shape[1]
    resp = tokenizer.decode(outputs[0][input_len:], skip_special_tokens=True).strip()
    print(f"[{idx+1}] 测试场景: {desc}")
    print(f"    客户端 Prompt: {prompt}")
    print(f"    模型响应:      {resp}")
    print()

print("=================================================================")
print("🚀 [4/4] 验证 LoRA Adapter 转 GGUF (供 llama.cpp / llama-server 使用)...")
print("=================================================================")
if not os.path.exists("llama.cpp"):
    print("⚙️ 拉取 llama.cpp 转换工具...")
    subprocess.run(["git", "clone", "--depth", "1", "https://github.com/ggml-org/llama.cpp"], check=True)

gguf_adapter_path = "output/gemma4-e2b-gec-lora-f16.gguf"
subprocess.run([
    "python3", "llama.cpp/convert_lora_to_gguf.py",
    lora_output_dir,
    "--outfile", gguf_adapter_path,
    "--outtype", "f16",
    "--base-model-id", target_base
], check=True)
print(f"🎉 GGUF LoRA 适配器导出成功！文件路径: {gguf_adapter_path} ({os.path.getsize(gguf_adapter_path)/1024/1024:.2f} MB)")


In [ ]:
#@title 🔑 [3/4] 配置 GitHub Token (用于自动发布 Release)
import os
try:
    from google.colab import userdata
    token = userdata.get('GH_TOKEN')
except Exception:
    token = None

if not token and not os.environ.get('GH_TOKEN') and not os.environ.get('GITHUB_TOKEN'):
    token = input("请输入你的 GitHub Token (按回车直接上传): ").strip()

if token:
    os.environ['GH_TOKEN'] = token
    os.environ['GITHUB_TOKEN'] = token
    print("✅ GitHub Token 配置成功！")
else:
    print("ℹ️ 未提供 Token，训练完成后模型将保存在 output 目录。")

In [ ]:
#@title 🚀 [4/4] 启动 Qwen3 Adapter 训练、量化与发布！
!chmod +x scripts/release_model.sh
import subprocess
import os

print(f"🧭 当前 mode: {mode}", flush=True)
print(f"🏷️ 当前版本: {version_tag}", flush=True)
print(f"🌿 当前 branch 参数: {branch}", flush=True)
subprocess.run(["git", "rev-parse", "--short", "HEAD"], check=True)
script_path = os.path.join(os.getcwd(), "scripts", "release_model.sh")
print(f"📄 release_model.sh: exists={os.path.exists(script_path)} size={os.path.getsize(script_path) if os.path.exists(script_path) else 0}", flush=True)
if os.path.exists(script_path):
    with open(script_path, encoding="utf-8") as script_file:
        print("📄 script head:", script_file.readline().strip(), flush=True)

def run(cmd):
    """逐行实时转发 stdout/stderr，长任务期间也持续显示进度。"""
    print(f"\n$ {cmd}", flush=True)
    print(f"📍 cwd: {os.getcwd()}", flush=True)
    process = subprocess.Popen(
        ["/bin/bash", "-o", "pipefail", "-c", cmd],
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        encoding="utf-8",
        bufsize=1,
    )
    assert process.stdout is not None
    for line in process.stdout:
        print(line, end="", flush=True)
    returncode = process.wait()
    print(f"↩️ exit code: {returncode}", flush=True)
    if returncode != 0:
        raise SystemExit(f"命令失败 (exit {returncode}): {cmd}")

task_list = ["gec", "completion", "distill"]

def release_full_matrix(target_tier, target_base_model):
    """每个用户可见 Tier 由一个 Base 与三个隐藏任务 Adapter 组成。"""
    run(f"./scripts/release_model.sh {version_tag} {target_base_model} --tier {target_tier} --asset base")
    for task in task_list:
        run(f"./scripts/release_model.sh {version_tag} {target_base_model} --tier {target_tier} --asset adapter --task {task}")

if "Lite 完整矩阵" in mode:
    release_full_matrix("lite", "Qwen/Qwen3-0.6B")
elif "Standard 完整矩阵 (Gemma 4 E2B)" in mode:
    release_full_matrix("standard", "google/gemma-4-E2B-it")
elif "Standard 完整矩阵" in mode:
    release_full_matrix("standard", "Qwen/Qwen3-1.7B")
elif "双矩阵" in mode:
    # 顺序运行，避免在单张 Colab GPU 上同时占用两个训练进程。
    release_full_matrix("lite", "Qwen/Qwen3-0.6B")
    release_full_matrix("standard", "Qwen/Qwen3-1.7B")
elif mode == "base":
    run(f"./scripts/release_model.sh {version_tag} {base_model} --tier {tier} --asset base")
elif mode == "legacy 完整模型(旧版兼容)":
    run(f"./scripts/release_model.sh {version_tag} {base_model}")
else:
    task = mode.split()[0]  # gec/completion/distill adapter
    run(f"./scripts/release_model.sh {version_tag} {base_model} --tier {tier} --asset adapter --task {task}")

print("🎉 全流程执行完毕！已上线 GitHub Releases！")
